# TrustOCT Colab Orchestrator
### Title: TrustOCT: A Trustworthy Retinal OCT Disease Classification Framework Using Domain Generalization and Uncertainty-Aware Selective Prediction

This notebook acts as the high-level Colab controller. All core code is modularized in standalone Python files. The cells in this notebook clone the repository, download datasets, and run training and evaluation wrappers.

### 1. Clone Repository & Install Requirements

In [ ]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive')

# Clone repo
!git clone https://github.com/Gnanapravallika/TrusthOCTAI.git
%cd TrusthOCTAI
!git pull origin main

# Install dependencies
!pip install -r requirements.txt
!pip install grad-cam

### 2. Dataset Preparation (Kermany OCT2017)

In [ ]:
import os

if not os.path.exists('/content/Kermany') and not os.path.exists('/content/OCT2017'):
    try:
        from google.colab import files
        print('Please upload your kaggle.json API token file:')
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            !mkdir -p ~/.kaggle
            !cp kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json
            !kaggle datasets download -d paultimothymooney/kermany2018 --unzip -p /content/Kermany
            print('Dataset downloaded successfully.')
    except Exception as e:
        print(f'Skipped: {e}')

# Create local symlinks for dataset compatibility
!mkdir -p datasets
if not os.path.exists('datasets/Kermany'):
    !ln -s /content/Kermany datasets/Kermany

# Copy OCTID dataset from Google Drive if mounted
if os.path.exists('/content/drive/MyDrive/OCTID'):
    !mkdir -p datasets/OCTID
    !cp -r "/content/drive/MyDrive/OCTID/." datasets/OCTID/
    print('OCTID dataset copied.')
else:
    print('Google Drive not mounted or OCTID folder missing. Skipping OCTID copy.')

### 3. Dataset Audit & Statistics (Phase 2 & 3)

In [ ]:
# Run verification status checks and compute pixel statistics
!python main.py --mode verify --experiment_name EXP003_CBAM

### 4. Execute Model Training (Phase 6)

In [ ]:
#@title Select Active Experiment Settings
EXPERIMENT_NAME = "EXP003_CBAM" #@param ["EXP001_Baseline", "EXP002_MultiScale", "EXP003_CBAM"]

# Copy target experiment model configuration
!cp configs/{EXPERIMENT_NAME}.yaml configs/model.yaml

# Execute training run
!python train.py --experiment_name {EXPERIMENT_NAME}

### 5. Run Evaluation & Generate Explanations (Phase 7, 8 & 9)

In [ ]:
# Evaluate checkpoints and generate reliability diagrams + LayerCAM heatmaps
!python test.py --experiment_name {EXPERIMENT_NAME}

# Print LayerCAM explainability overlays directly in notebook
from IPython.display import Image, display
overlay_path = f"outputs/{EXPERIMENT_NAME}/explainability_check_layercam.png"
if os.path.exists(overlay_path):
    print("LayerCAM Visual Attribution Overlay:")
    display(Image(filename=overlay_path))

### 6. Single Scan Inference & Uncertainty Check

In [ ]:
#@title Test Prediction on a Sample File
IMAGE_PATH = "datasets/Kermany/OCT2017 /test/NORMAL/NORMAL-1017326-1.jpeg" #@param {type:"string"}

!python inference.py --image_path {IMAGE_PATH} --weights_path outputs/{EXPERIMENT_NAME}/weights_best.pth

### 7. Export Outputs to Google Drive

In [ ]:
# Zip assets and copy to Drive persistently
!zip -r outputs.zip outputs/
!cp outputs.zip "/content/drive/MyDrive/TrustOCT_outputs.zip"
print("[OK] Zipped all checkpoints, metrics, and calibration plots to Drive.")